No,４

In [1]:
import os
import re
import tkinter as tk
from tkinter import filedialog
from pathlib import Path

拡張子ごとにフォルダ分けしたファイルの名前を「0+元のファイル名から抽出した連番数字_サブサブフォルダ名_サブフォルダ名」に変更

In [2]:
#指定したメインフォルダ内のすべてのファイル名を「0+抽出した連番数字_サブサブフォルダ名_サブフォルダ名」となるように変更


def rename_csv_files_flexible():
    # 1. フォルダ選択ダイアログの表示設定
    root = tk.Tk()
    root.attributes('-topmost', True) 
    root.withdraw()

    # 一番上にある大元のフォルダを選択
    top_dir_path = filedialog.askdirectory(title="一番上の大元フォルダを選択してください")
    
    if not top_dir_path:
        print("フォルダ選択がキャンセルされました。処理を終了します。")
        return

    top_dir = Path(top_dir_path)
    print(f"選択された大元フォルダ: {top_dir.name}\n")

    rename_count = 0
    
    # 2. rglob("*.csv") で、階層の深さに関係なく大元フォルダ内の全CSVファイルを取得
    for file_path in top_dir.rglob("*.csv"):
        original_name = file_path.name
        
        # 3. ファイルから見て「末端のフォルダ」と「末端の前のフォルダ」の名前を取得
        terminal_folder = file_path.parent.name         # 末端のフォルダ名
        pre_terminal_folder = file_path.parent.parent.name # 末端の前のフォルダ名
        
        # 4. 正規表現で「$」の後の連番数字を抽出
        match = re.search(r'\$(\d+)\.csv$', original_name)
        
        if match:
            seq_num = match.group(1) # 抽出した連番数字
            
            # 5. 新しいファイル名の作成
            # 修正箇所: 0 + 抽出した連番数字 + _ + 末端の前のフォルダ名 + _ + 末端のフォルダ名 + .csv
            new_name = f"0{seq_num}_{pre_terminal_folder}_{terminal_folder}.csv"
            new_file_path = file_path.parent / new_name
            
            # 6. ファイル名の変更を実行
            try:
                file_path.rename(new_file_path)
                print(f"変更完了: {pre_terminal_folder}/{terminal_folder}/{original_name} -> {new_name}")
                rename_count += 1
            except Exception as e:
                print(f"エラー ({original_name}): {e}")

    print(f"\n処理が完了しました。合計 {rename_count} 件のファイル名を変更しました。")

# 関数の実行
rename_csv_files_flexible()

選択された大元フォルダ: csv

変更完了: １９０℃/０１０/010$10.csv -> 010_１９０℃_０１０.csv
変更完了: １９０℃/０１０/010$11.csv -> 011_１９０℃_０１０.csv
変更完了: １９０℃/０１０/010$2.csv -> 02_１９０℃_０１０.csv
変更完了: １９０℃/０１０/010$3.csv -> 03_１９０℃_０１０.csv
変更完了: １９０℃/０１０/010$4.csv -> 04_１９０℃_０１０.csv
変更完了: １９０℃/０１０/010$5.csv -> 05_１９０℃_０１０.csv
変更完了: １９０℃/０１０/010$6.csv -> 06_１９０℃_０１０.csv
変更完了: １９０℃/０１０/010$7.csv -> 07_１９０℃_０１０.csv
変更完了: １９０℃/０１０/010$8.csv -> 08_１９０℃_０１０.csv
変更完了: １９０℃/０１０/010$9.csv -> 09_１９０℃_０１０.csv
変更完了: １９０℃/０２０/020$10.csv -> 010_１９０℃_０２０.csv
変更完了: １９０℃/０２０/020$11.csv -> 011_１９０℃_０２０.csv
変更完了: １９０℃/０２０/020$2.csv -> 02_１９０℃_０２０.csv
変更完了: １９０℃/０２０/020$3.csv -> 03_１９０℃_０２０.csv
変更完了: １９０℃/０２０/020$4.csv -> 04_１９０℃_０２０.csv
変更完了: １９０℃/０２０/020$5.csv -> 05_１９０℃_０２０.csv
変更完了: １９０℃/０２０/020$6.csv -> 06_１９０℃_０２０.csv
変更完了: １９０℃/０２０/020$7.csv -> 07_１９０℃_０２０.csv
変更完了: １９０℃/０２０/020$8.csv -> 08_１９０℃_０２０.csv
変更完了: １９０℃/０２０/020$9.csv -> 09_１９０℃_０２０.csv
変更完了: １９０℃/０４０/040$10.csv -> 010_１９０℃_０４０.csv
変更完了: １９０℃/０４０/040$11.csv -> 011_１９０℃_０４０.csv
変更

ファイル名変更後のファイルの移動と、ファイルが入っていた末端フォルダの削除

In [3]:
#ファイル名変更後、末端フォルダの削除と末端フォルダの前フォルダに名前変更後ファイルの移動
#移動後に連番の振り直し

def reorganize_resort_and_rename_001():
    # 1. フォルダ選択ダイアログの表示設定
    root = tk.Tk()
    root.attributes('-topmost', True) 
    root.withdraw()

    # 一番上にある大元のフォルダを選択
    top_dir_path = filedialog.askdirectory(title="一番上の大元フォルダを選択してください")
    
    if not top_dir_path:
        print("フォルダ選択がキャンセルされました。処理を終了します。")
        return

    top_dir = Path(top_dir_path)
    print(f"選択された大元フォルダ: {top_dir.name}\n")

    pre_term_files = {}
    folders_to_delete = set()

    # ==========================================
    # 手順1: ファイルの抽出と末端前フォルダへの移動
    # ==========================================
    for file_path in top_dir.rglob("*.csv"):
        terminal_folder = file_path.parent
        pre_terminal_folder = file_path.parent.parent
        
        if terminal_folder == top_dir or pre_terminal_folder == top_dir.parent:
            continue
            
        try:
            temp_move_path = pre_terminal_folder / f"temp_{file_path.name}"
            file_path.rename(temp_move_path)
            
            folders_to_delete.add(terminal_folder)
            
            if pre_terminal_folder not in pre_term_files:
                pre_term_files[pre_terminal_folder] = []
            pre_term_files[pre_terminal_folder].append(temp_move_path)
            
        except Exception as e:
            print(f"移動エラー ({file_path.name}): {e}")

    # ==========================================
    # 手順2: 後ろの数字によるソートと、連番の振り直し
    # ==========================================
    for pre_term_dir, files in pre_term_files.items():
        sortable_list = []
        
        for temp_path in files:
            original_name = temp_path.name.replace("temp_", "", 1)
            
            match = re.match(r'^(\d+)_(.*)_(\d+)\.csv$', original_name)
            
            if match:
                front_str = match.group(1)
                middle_str = match.group(2)
                back_str = match.group(3)
                
                front_num = int(front_str)
                back_num = int(back_str)
                
                sortable_list.append({
                    'back_num': back_num,
                    'front_num': front_num,
                    'front_str': front_str,
                    'middle_str': middle_str,
                    'back_str': back_str,
                    'temp_path': temp_path
                })
            else:
                print(f"警告: ファイル名形式が一致しません。リネームをスキップします: {original_name}")
                temp_path.rename(pre_term_dir / original_name)

        sortable_list.sort(key=lambda x: (x['back_num'], x['front_num']))

        # === 変更箇所：1からスタートし、3桁の0埋めに設定 ===
        # enumerate(..., start=1) でカウントを1から開始します
        for i, item in enumerate(sortable_list, start=1):
            # :03d を指定することで「001」「002」...という3桁のフォーマットになります
            new_front = f"{i:03d}"
            
            new_name = f"{new_front}_{item['middle_str']}_{item['back_str']}.csv"
            final_path = pre_term_dir / new_name
            
            try:
                item['temp_path'].rename(final_path)
                print(f"整理完了: {pre_term_dir.name}/{new_name}")
            except Exception as e:
                print(f"リネームエラー: {e}")

    # ==========================================
    # 手順3: 空になった末端フォルダの削除
    # ==========================================
    print("\n--- フォルダの整理を開始します ---")
    deleted_folder_count = 0
    
    for folder in folders_to_delete:
        ds_store = folder / ".DS_Store"
        if ds_store.exists():
            ds_store.unlink()
            
        try:
            folder.rmdir()
            deleted_folder_count += 1
        except OSError:
            print(f"スキップ: {folder.name} (内部にCSV以外のファイルが残っているため保持しました)")

    print(f"\nすべての処理が完了しました！")
    print(f"{deleted_folder_count} 個の不要な末端フォルダを削除しました。")

# 関数の実行
reorganize_resort_and_rename_001()

選択された大元フォルダ: csv

整理完了: １９０℃/001_１９０℃_０１０.csv
整理完了: １９０℃/002_１９０℃_０１０.csv
整理完了: １９０℃/003_１９０℃_０１０.csv
整理完了: １９０℃/004_１９０℃_０１０.csv
整理完了: １９０℃/005_１９０℃_０１０.csv
整理完了: １９０℃/006_１９０℃_０１０.csv
整理完了: １９０℃/007_１９０℃_０１０.csv
整理完了: １９０℃/008_１９０℃_０１０.csv
整理完了: １９０℃/009_１９０℃_０１０.csv
整理完了: １９０℃/010_１９０℃_０１０.csv
整理完了: １９０℃/011_１９０℃_０２０.csv
整理完了: １９０℃/012_１９０℃_０２０.csv
整理完了: １９０℃/013_１９０℃_０２０.csv
整理完了: １９０℃/014_１９０℃_０２０.csv
整理完了: １９０℃/015_１９０℃_０２０.csv
整理完了: １９０℃/016_１９０℃_０２０.csv
整理完了: １９０℃/017_１９０℃_０２０.csv
整理完了: １９０℃/018_１９０℃_０２０.csv
整理完了: １９０℃/019_１９０℃_０２０.csv
整理完了: １９０℃/020_１９０℃_０２０.csv
整理完了: １９０℃/021_１９０℃_０４０.csv
整理完了: １９０℃/022_１９０℃_０４０.csv
整理完了: １９０℃/023_１９０℃_０４０.csv
整理完了: １９０℃/024_１９０℃_０４０.csv
整理完了: １９０℃/025_１９０℃_０４０.csv
整理完了: １９０℃/026_１９０℃_０４０.csv
整理完了: １９０℃/027_１９０℃_０４０.csv
整理完了: １９０℃/028_１９０℃_０４０.csv
整理完了: １９０℃/029_１９０℃_０４０.csv
整理完了: １９０℃/030_１９０℃_０４０.csv
整理完了: １９０℃/031_１９０℃_０８０.csv
整理完了: １９０℃/032_１９０℃_０８０.csv
整理完了: １９０℃/033_１９０℃_０８０.csv
整理完了: １９０℃/034_１９０℃_０８０.csv
整理完了: １９０℃/035_１９０℃_０８０.csv
整理